# Quantum Teleportation and Noise Analysis with Qiskit

**A Colab-ready project with two execution paths:**
1. **Qiskit Aer simulator** — runs on the Colab CPU and does not require an IBM Quantum account.
2. **IBM Quantum hardware** — optional execution on a real quantum processor when account access and a compatible backend are available.

The notebook first verifies teleportation in an ideal simulator, then studies how gate and measurement noise affect the verification fidelity. If IBM Quantum access is available, the same protocol can also be submitted to hardware.

> **Security:** Never paste an IBM Quantum API token into a notebook or commit it to GitHub. For optional hardware access, use Google Colab Secrets as described below.

## Project objectives

- Prepare three known input states: \(|0\rangle\), \(|1\rangle\), and \(|+\rangle\).
- Build a three-qubit teleportation circuit using Alice's two qubits and Bob's qubit.
- Verify Bob's received state by applying the inverse of the state-preparation operation before measuring.
- Run ideal and noisy simulations using Qiskit Aer.
- Estimate verification fidelity for different noise strengths.
- Optionally compare the result with a real IBM Quantum backend.

## How the two execution methods fit together

| Method | Requires IBM account? | Purpose |
|---|---:|---|
| Qiskit Aer ideal simulator | No | Baseline correctness check |
| Qiskit Aer noisy simulator | No | Controlled noise analysis |
| IBM Quantum hardware (optional) | Yes | Real-device experiment |

**Recommended path:** Run the complete Aer workflow first. The IBM section is optional and is disabled by default, so the notebook remains usable without IBM Quantum access.


## 1. Install dependencies

Run this cell in Google Colab. It installs Qiskit, Qiskit Aer, plotting/data packages, and the optional IBM Runtime client.

If Colab asks for a runtime restart after installation, restart the session and then continue from the imports cell.

In [ ]:
%pip -q install "qiskit>=2.1.0" "qiskit-aer>=0.17.0" "qiskit-ibm-runtime>=0.40.1" numpy pandas matplotlib pylatexenc

## 2. Imports and configuration

We use a fixed number of shots so that runs are reproducible in scale. Measurement outcomes are still sampled, so small statistical fluctuations are expected.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

from IPython.display import display

SEED = 42
SHOTS = 4096

np.random.seed(SEED)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Qiskit and Qiskit Aer imports completed.")
print(f"Shots per circuit: {SHOTS}")
print(f"Results directory: {RESULTS_DIR.resolve()}")

## 3. Define the input quantum states

A general pure qubit state can be written as

\[
|\psi\rangle = \cos\left(\frac{\theta}{2}\right)|0\rangle
+ e^{i\phi}\sin\left(\frac{\theta}{2}\right)|1\rangle.
\]

Qiskit's `u(theta, phi, lambda, qubit)` gate prepares this state from \(|0\rangle\) when \(\lambda=0\).

We use three test states:

- \(|0\rangle\): \(\theta=0,\ \phi=0\)
- \(|1\rangle\): \(\theta=\pi,\ \phi=0\)
- \(|+\rangle=(|0\rangle+|1\rangle)/\sqrt{2}\): \(\theta=\pi/2,\ \phi=0\)

These are known test states, not arbitrary unknown states. The inverse-preparation verification method below is therefore suitable for this controlled experiment.

In [ ]:
quantum_states = {
    "|0>": (0.0, 0.0),
    "|1>": (np.pi, 0.0),
    "|+>": (np.pi / 2, 0.0),
}

state_table = pd.DataFrame(
    [
        {"state": name, "theta": theta, "phi": phi}
        for name, (theta, phi) in quantum_states.items()
    ]
)
display(state_table)

## 4. Build the quantum teleportation circuit

Qubit roles:

| Qubit | Role |
|---|---|
| `q[0]` | Alice's input (secret) qubit |
| `q[1]` | Alice's half of the Bell pair |
| `q[2]` | Bob's half of the Bell pair |

Classical-bit roles:

| Bit | Stores |
|---|---|
| `c[0]` | Alice's input-qubit measurement |
| `c[1]` | Alice's Bell-pair-qubit measurement |
| `c[2]` | Bob's verification measurement |

Protocol:
1. Create a Bell pair between Alice and Bob.
2. Prepare the input state on Alice's secret qubit.
3. Perform Alice's Bell-basis operations and measurements.
4. Apply Bob's classically controlled corrections: `X` if `c[1] == 1`, then `Z` if `c[0] == 1`.
5. Apply the inverse state-preparation gate to Bob and measure him.

The final measurement should ideally be `0` for Bob's verification bit. Qiskit displays the highest-index classical bit on the left, so for the single register `c`, the leftmost bit in a count string is `c[2]`.

In [ ]:
def create_teleportation_circuit(theta, phi):
    """Create a teleportation circuit and verify Bob's output.

    q[0]: Alice's input qubit
    q[1]: Alice's entangled qubit
    q[2]: Bob's qubit

    c[0], c[1]: Alice's measurement results
    c[2]: Bob's verification measurement
    """
    q = QuantumRegister(3, "q")
    c = ClassicalRegister(3, "c")
    qc = QuantumCircuit(q, c)

    secret = q[0]
    alice = q[1]
    bob = q[2]

    # 1. Alice and Bob share an entangled Bell pair.
    qc.h(alice)
    qc.cx(alice, bob)
    qc.barrier()

    # 2. Prepare the input state on Alice's secret qubit.
    qc.u(theta, phi, 0.0, secret)
    qc.barrier()

    # 3. Alice performs Bell-basis operations.
    qc.cx(secret, alice)
    qc.h(secret)
    qc.barrier()

    # 4. Alice measures her two qubits.
    qc.measure(secret, c[0])
    qc.measure(alice, c[1])

    # 5. Bob applies corrections based on Alice's classical results.
    with qc.if_test((c[1], 1)):
        qc.x(bob)
    with qc.if_test((c[0], 1)):
        qc.z(bob)

    qc.barrier()

    # 6. Undo the known state preparation and verify Bob's state.
    qc.u(theta, phi, 0.0, bob).inverse()
    qc.measure(bob, c[2])

    return qc


# Build and display an example circuit for |+>.
example_qc = create_teleportation_circuit(*quantum_states["|+>"])
print(example_qc.draw(output="text"))

## 5. Run an ideal simulation with Qiskit Aer

This is the fallback execution method when IBM Quantum hardware is unavailable. Aer runs the circuit on the Colab CPU; no API key or cloud quantum processor is needed.

We first run without a noise model. Ideally, Bob's verification bit is always zero.

In [ ]:
ideal_simulator = AerSimulator()

def run_aer_circuit(circuit, simulator, shots=SHOTS, seed=SEED):
    """Transpile and execute a circuit on an Aer simulator; return counts."""
    compiled_circuit = transpile(
        circuit,
        simulator,
        optimization_level=0,
        seed_transpiler=seed,
    )
    job = simulator.run(compiled_circuit, shots=shots, seed_simulator=seed)
    result = job.result()
    return result.get_counts()


def verification_fidelity(counts):
    """Estimate verification fidelity from Bob's final classical bit c[2].

    With one 3-bit classical register, Qiskit displays c[2] as the
    leftmost bit. A zero in that position is a successful verification.
    """
    total_shots = sum(counts.values())
    if total_shots == 0:
        raise ValueError("No measurement shots were returned.")

    successful_shots = 0
    for bitstring, count in counts.items():
        clean_bitstring = bitstring.replace(" ", "")
        bob_verification_bit = clean_bitstring[0]
        if bob_verification_bit == "0":
            successful_shots += count

    return successful_shots / total_shots


ideal_results = []
ideal_counts_by_state = {}

for state_name, (theta, phi) in quantum_states.items():
    circuit = create_teleportation_circuit(theta, phi)
    counts = run_aer_circuit(circuit, ideal_simulator)
    fidelity = verification_fidelity(counts)

    ideal_counts_by_state[state_name] = counts
    ideal_results.append({
        "state": state_name,
        "shots": sum(counts.values()),
        "verification_fidelity": fidelity,
    })

ideal_df = pd.DataFrame(ideal_results)
display(ideal_df)

print("Example counts for |+>:", ideal_counts_by_state["|+>"])

### Interpreting the verification fidelity

For a pure target state, fidelity is

\[
F = |\langle\psi_{\mathrm{target}}|\psi_{\mathrm{received}}\rangle|^2.
\]

In this notebook, we apply the inverse of the known preparation gate to Bob's qubit and measure it. A zero result means that Bob's state passed this verification test. Thus,

\[
F_{\mathrm{estimated}} =
\frac{\text{number of shots with }c[2]=0}
{\text{total number of shots}}.
\]

This is a verification-based estimate for the known test states. It is **not** full quantum state tomography. In an ideal simulator, the result should be 1 (up to numerical or implementation issues); with finite shots, the verification outcome is expected to be deterministic for this ideal circuit.

## 6. Define the noise model

We model two sources of error:

1. **Depolarizing gate noise** on single-qubit `u` gates and two-qubit `cx` gates.
2. **Symmetric readout error**, where a measured 0 can be reported as 1 and vice versa, each with probability `p`.

The parameter `p` controls the strength of the model. It is not a direct prediction of the final infidelity. Noise channels are applied only to matching instructions in the transpiled circuit.

In [ ]:
def create_noise_model(p):
    """Create a simple depolarizing + symmetric readout noise model.

    p must be between 0 and 1. The same p is used as a convenient
    experiment control for gate and readout error probabilities.
    """
    if not 0.0 <= p <= 1.0:
        raise ValueError("Noise probability p must be between 0 and 1.")

    noise_model = NoiseModel()

    one_qubit_error = depolarizing_error(p, 1)
    two_qubit_error = depolarizing_error(p, 2)

    # Apply gate noise to the circuit's U and CX instructions.
    noise_model.add_all_qubit_quantum_error(one_qubit_error, ["u"])
    noise_model.add_all_qubit_quantum_error(two_qubit_error, ["cx"])

    # Symmetric measurement bit-flip error:
    # P(0->1) = p and P(1->0) = p.
    readout_error = ReadoutError([
        [1.0 - p, p],
        [p, 1.0 - p],
    ])
    noise_model.add_all_qubit_readout_error(readout_error)

    return noise_model


# Inspect a small example noise model.
example_noise_model = create_noise_model(0.01)
print("Noise model created.")
print("Noise instructions:", example_noise_model.noise_instructions)
print("Noise basis gates:", example_noise_model.basis_gates)

## 7. Run noise analysis with Qiskit Aer

We sweep several noise probabilities and execute each of the three input states at every noise level.

For each run:
- Build the teleportation circuit.
- Attach a noise model to an Aer simulator.
- Execute the circuit.
- Estimate Bob's verification fidelity from `c[2]`.

The experiment stores both state-wise fidelity and the mean across the three test states.

In [ ]:
noise_levels = [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]

noise_results = []

for p in noise_levels:
    noise_model = create_noise_model(p)
    noisy_simulator = AerSimulator(noise_model=noise_model)

    for state_name, (theta, phi) in quantum_states.items():
        circuit = create_teleportation_circuit(theta, phi)
        counts = run_aer_circuit(circuit, noisy_simulator)
        fidelity = verification_fidelity(counts)

        noise_results.append({
            "noise_probability": p,
            "state": state_name,
            "shots": sum(counts.values()),
            "verification_fidelity": fidelity,
        })

noise_df = pd.DataFrame(noise_results)
display(noise_df.head(9))

In [ ]:
# Aggregate the state-wise results at each noise level.
mean_noise_df = (
    noise_df.groupby("noise_probability", as_index=False)
    ["verification_fidelity"]
    .mean()
    .rename(columns={"verification_fidelity": "mean_verification_fidelity"})
)

display(mean_noise_df)

## 8. Visualize the noise analysis

The first plot shows the verification fidelity for each input state. The second plot shows the mean fidelity across the three test states.

In [ ]:
plt.figure(figsize=(9, 5))
for state_name in quantum_states:
    state_data = noise_df[noise_df["state"] == state_name]
    plt.plot(
        state_data["noise_probability"],
        state_data["verification_fidelity"],
        marker="o",
        label=state_name,
    )

plt.xlabel("Noise probability (p)")
plt.ylabel("Verification fidelity")
plt.title("Quantum Teleportation: Fidelity vs Noise")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(title="Input state")
plt.tight_layout()

state_plot_path = RESULTS_DIR / "fidelity_by_state_vs_noise.png"
plt.savefig(state_plot_path, dpi=200, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(
    mean_noise_df["noise_probability"],
    mean_noise_df["mean_verification_fidelity"],
    marker="o",
)
plt.xlabel("Noise probability (p)")
plt.ylabel("Mean verification fidelity")
plt.title("Mean Teleportation Fidelity vs Noise")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.tight_layout()

mean_plot_path = RESULTS_DIR / "mean_fidelity_vs_noise.png"
plt.savefig(mean_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", state_plot_path)
print("Saved:", mean_plot_path)

## 9. Export results

The CSV files can be committed to GitHub after you have run the notebook. They provide a record of the actual simulator output from your run.

In [ ]:
noise_csv_path = RESULTS_DIR / "noise_analysis_results.csv"
ideal_csv_path = RESULTS_DIR / "ideal_simulation_results.csv"

noise_df.to_csv(noise_csv_path, index=False)
ideal_df.to_csv(ideal_csv_path, index=False)

print("Saved:", noise_csv_path)
print("Saved:", ideal_csv_path)

print("\nIdeal simulation summary:")
display(ideal_df)

print("\nMean fidelity by noise level:")
display(mean_noise_df)

## 10. Optional: Run on IBM Quantum hardware

This section is optional. The Aer simulator workflow above is complete and does not depend on IBM Quantum.

### Configure Colab Secrets

1. Open the **Secrets** panel in Google Colab (key icon in the left sidebar).
2. Add a secret named `IBM_QUANTUM_TOKEN`.
3. Paste your newly generated IBM Quantum API token as its value.
4. Enable notebook access to that secret.

Do not write the token directly into a code cell or save it in the notebook.

The hardware path uses Qiskit IBM Runtime's `SamplerV2`. A compatible backend must support dynamic circuits / classical feed-forward because teleportation requires Bob's correction gates to depend on Alice's measurement results. Availability and backend support can change. If execution fails due to backend capability or account access, continue using Aer.

Set `RUN_IBM = True` only when you are ready to submit a real hardware job. Hardware jobs may consume account resources or quota.

In [ ]:
# Optional IBM Quantum hardware execution.
# Keep False to skip this section and use Aer only.
RUN_IBM = False

if RUN_IBM:
    try:
        from google.colab import userdata
        from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
        from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

        ibm_token = userdata.get("IBM_QUANTUM_TOKEN")
        if not ibm_token:
            raise ValueError(
                "Add IBM_QUANTUM_TOKEN in Colab Secrets and enable notebook access."
            )

        service = QiskitRuntimeService(
            channel="ibm_quantum_platform",
            token=ibm_token,
        )

        backend = service.least_busy(
            operational=True,
            simulator=False,
            min_num_qubits=3,
        )
        print("Selected IBM backend:", backend.name)

        # Try the |+> test state as a first hardware demonstration.
        hardware_circuit = create_teleportation_circuit(
            *quantum_states["|+>"]
        )

        pass_manager = generate_preset_pass_manager(
            backend=backend,
            optimization_level=1,
        )
        isa_circuit = pass_manager.run(hardware_circuit)

        sampler = SamplerV2(mode=backend)
        hardware_job = sampler.run([isa_circuit], shots=SHOTS)
        print("Submitted IBM job:", hardware_job.job_id())

        hardware_result = hardware_job.result()
        hardware_counts = hardware_result[0].data.c.get_counts()
        hardware_fidelity = verification_fidelity(hardware_counts)

        print("Hardware counts:", hardware_counts)
        print(f"Hardware verification fidelity: {hardware_fidelity:.4f}")

    except Exception as exc:
        print("IBM hardware execution was not completed.")
        print("Reason:", repr(exc))
        print(
            "You can still complete the project using the Aer ideal and noisy "
            "simulation results generated above."
        )
else:
    print("IBM hardware section skipped. Aer simulation remains fully available.")

## 11. Interpretation and limitations

Use your generated tables and plots to discuss the observations. Do not assume a particular numerical result before running the notebook.

Questions to consider:
- Does the ideal simulator produce verification fidelity close to 1 for all three states?
- How does the mean verification fidelity change as `p` increases?
- Do the three input states respond identically to this particular noise model?
- How much of the observed degradation comes from gate errors versus readout errors?

### Important limitations

- The depolarizing and readout noise models are simplified mathematical models, not a calibrated description of a specific IBM processor.
- The same `p` is used as a convenient control parameter for different error channels. It should not be interpreted as a measured hardware error rate.
- The verification method uses known input states and inverse state preparation. It is not full state tomography and does not characterize an arbitrary unknown output state.
- The IBM hardware path is optional and depends on account access, backend availability, and dynamic-circuit support.
- Hardware results can differ from Aer results because real devices have time-varying noise, connectivity restrictions, calibration differences, and finite sampling.

## 12. Suggested GitHub repository structure

After running the notebook, your repository can look like this:

```text
quantum-teleportation-noise-analysis/
│
├── Quantum_Teleportation_Noise_Analysis.ipynb
├── README.md
├── requirements.txt
├── .gitignore
│
└── results/
    ├── fidelity_by_state_vs_noise.png
    ├── mean_fidelity_vs_noise.png
    ├── noise_analysis_results.csv
    └── ideal_simulation_results.csv
```

## 13. References

1. Bennett et al. (1993), *Teleporting an Unknown Quantum State via Dual Classical and Einstein-Podolsky-Rosen Channels*, Physical Review Letters. https://doi.org/10.1103/PhysRevLett.70.1895
2. Qiskit documentation: https://docs.quantum.ibm.com/
3. Qiskit Aer documentation: https://qiskit.github.io/qiskit-aer/
4. Qiskit IBM Runtime documentation: https://quantum.cloud.ibm.com/docs/en/guides

---

**Author:** Pranava Prakash J  
**Project:** Quantum Teleportation and Noise Analysis Using Qiskit  
**Purpose:** Educational project exploring quantum teleportation, simulation, noise, and optional real-device execution.
